# Chapter 8: AI-Powered Equity Investing

Companion notebook for Chapter 8 of *Mastering AI-Powered Finance* (`Overleaf/Book 1`). It was built from Chapter 7 of the full manuscript (Equity Investing and Portfolio Management); the source references below point to that manuscript.

Notebook extracted from `Overleaf/Final Project/Chapter 7 - Equity Investing and Portfolio Management/main.tex`.

This notebook turns the chapter code examples into a student-facing workflow. Code cells follow the order used in the chapter; Markdown cells identify the source sections and explain how to interpret the computed output.

Run the notebook from top to bottom. Some examples are intentionally small and synthetic; examples that call external services require the dependency or credential noted below.

**Data:** prices and fundamentals load from the bundled snapshots in `data/`, so the notebook runs offline and reproduces the figures in the book. Set `REFRESH_MARKET_DATA=1` to fetch current provider data instead, and expect different results. The optional Fama-French example needs internet access and the pandas-datareader package.

**Review notes:** The fundamentals snapshot is dated 21 September 2026 and supports an illustrative cross-sectional ranking only. Because that date falls after the 2019-2023 price window, it is not point-in-time information for the backtest period. The default backtest uses only lagged price factors (momentum and low volatility). A callable supplying as-reported fundamentals known at each signal date can enable the full factor model. The fixed stock universe still has survivorship bias. Month-end execution after a full month of decision lag, a constant risk-free assumption, simple linear trading costs, and frictionless cash are teaching assumptions, so the results are not evidence of investable alpha. Unverified historical output tables were removed; compute results from the code. Missing price observations are not filled, and held assets with missing realized returns raise an error. Supply complete calendar months for evaluation.

The original, more detailed workflow is preserved, with corrections to data handling, timing, trading costs, optimization checks, risk limits, and metrics.
Manuscript synchronization (2026-09-21): concise printed algorithms now match this expanded notebook. Obsolete output tables were removed. With the bundled snapshots in place the results are reproducible and do match the figures printed in the chapter.


In [1]:
# Common notebook setup
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
# Grayscale defaults, matching the printed book: black, mid and light gray,
# each with its own line style so series stay distinguishable in print.
from cycler import cycler
plt.rcParams['axes.prop_cycle'] = cycler(color=['0.0', '0.45', '0.7'], linestyle=['-', '--', ':'])
plt.rcParams['image.cmap'] = 'Greys'

## AI Tutorial: Stock Selection and Portfolio Construction

### Optional environment installation

```bash
pip install numpy pandas yfinance scipy scikit-learn matplotlib pandas-datareader
```


## AI Tutorial: Stock Selection and Portfolio Construction / Environment Setup and Data Acquisition

### Code example 1
Source line: `Overleaf/Final Project/Chapter 7 - Equity Investing and Portfolio Management/main.tex`

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize
from scipy import stats
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings


# Define a fixed illustrative investment universe: 52 stocks across sectors.
# Historical membership is not reconstructed, so survivorship bias remains.
UNIVERSE = [
    # Technology
    'AAPL', 'MSFT', 'GOOGL', 'META', 'NVDA', 'AVGO', 'CSCO', 'ORCL',
    # Healthcare
    'JNJ', 'UNH', 'PFE', 'ABBV', 'MRK', 'LLY', 'TMO', 'ABT',
    # Financials
    'JPM', 'BAC', 'WFC', 'GS', 'MS', 'BLK', 'AXP', 'C',
    # Consumer
    'AMZN', 'HD', 'MCD', 'NKE', 'SBUX', 'TGT', 'COST', 'WMT',
    # Industrials
    'CAT', 'HON', 'UPS', 'RTX', 'LMT', 'GE', 'MMM', 'BA',
    # Energy & Materials
    'XOM', 'CVX', 'COP', 'SLB', 'LIN', 'APD', 'ECL', 'SHW',
    # Utilities & REITs
    'NEE', 'DUK', 'SO', 'PLD'
]

# Benchmark: S&P 500 ETF
BENCHMARK = 'SPY'

def download_stock_data(tickers, start_date, end_date):
    """
    Download adjusted close prices for a list of tickers.

    Parameters:
    -----------
    tickers : list
        Stock ticker symbols
    start_date, end_date : str
        Date range in 'YYYY-MM-DD' format

    Returns:
    --------
    pd.DataFrame : Adjusted close prices, stocks as columns
    """
    all_tickers = list(dict.fromkeys(list(tickers) + [BENCHMARK]))
    snapshot = Path('data/prices_2019_2024.csv')
    if snapshot.exists() and os.environ.get('REFRESH_MARKET_DATA') != '1':
        data = pd.read_csv(snapshot, index_col=0, parse_dates=True)
        data = data.reindex(columns=all_tickers).sort_index()
    else:
        raw = yf.download(all_tickers, start=start_date, end=end_date,
                          auto_adjust=False, progress=False)
        if raw is None or raw.empty or 'Adj Close' not in raw.columns.get_level_values(0):
            raise ValueError("No adjusted prices returned; check the data service.")
        data = raw['Adj Close'].reindex(columns=all_tickers).sort_index()
    if data[BENCHMARK].dropna().empty:
        raise ValueError("Benchmark adjusted prices are unavailable.")
    if (data <= 0).any().any() or np.isinf(data.to_numpy()).any():
        raise ValueError("Adjusted prices must be positive and finite where observed.")
    # Do not select stocks using missingness over the full future test period.
    # Each rebalance will use only its trailing history to establish eligibility.
    return data

# Download 5 years of data for backtesting
END_DATE = '2024-01-01'
START_DATE = '2019-01-01'

print("Downloading stock data...")
prices = download_stock_data(UNIVERSE, START_DATE, END_DATE)
print(f"Downloaded data for {len(prices.columns)-1} stocks plus benchmark")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"Total trading days: {len(prices)}")

Downloaded data for 52 stocks plus benchmark
Date range: 2019-01-02 to 2023-12-29
Total trading days: 1258


### Reading the computed output

Run the preceding cell for the current calculation. The old manuscript table has been removed because it was not generated by the corrected workflow and contained inconsistent rankings or arithmetic. The current fundamental snapshot is a mechanics demonstration; historical results use lagged price factors unless an as-reported fundamentals callback is supplied.


## AI Tutorial: Stock Selection and Portfolio Construction / Environment Setup and Data Acquisition / Computing Returns and Fundamental Data

### Code example 2
Source line: `Overleaf/Final Project/Chapter 7 - Equity Investing and Portfolio Management/main.tex`

In [3]:
def compute_returns(prices):
    """Compute daily and monthly returns from prices."""
    daily_returns = prices.pct_change(fill_method=None).iloc[1:]

    # Monthly returns (last trading day of each month)
    # Keep the actual last row: resample().last() would skip missing closes.
    monthly_prices = prices.resample(pd.offsets.MonthEnd()).agg(lambda x: x.iloc[-1] if len(x) else np.nan)
    monthly_returns = monthly_prices.pct_change(fill_method=None).iloc[1:]

    return daily_returns, monthly_returns

daily_returns, monthly_returns = compute_returns(prices)

def get_fundamental_data(tickers):
    """
    Retrieve fundamental data for factor construction.

    Returns DataFrame with: P/E, P/B, ROE, Debt/Equity, Market Cap
    """
    # A pinned snapshot with a stated as-of date. yf.Ticker().info returns
    # whatever the market looks like today, with no historical as-of, so a
    # live fetch silently changes every ranking and weight below.
    snapshot = Path('data/fundamentals_snapshot.csv')
    if snapshot.exists() and os.environ.get('REFRESH_MARKET_DATA') != '1':
        return pd.read_csv(snapshot, index_col='ticker')

    fundamentals = []

    for ticker in tickers:
        if ticker == BENCHMARK:
            continue
        try:
            stock = yf.Ticker(ticker)
            info = stock.info

            fundamentals.append({
                'ticker': ticker,
                'pe_ratio': info.get('trailingPE', np.nan),
                'pb_ratio': info.get('priceToBook', np.nan),
                'roe': info.get('returnOnEquity', np.nan),
                'debt_to_equity': info.get('debtToEquity', np.nan),
                'market_cap': info.get('marketCap', np.nan),
                'dividend_yield': info.get('dividendYield', np.nan),
                'beta': info.get('beta', np.nan),
                'profit_margin': info.get('profitMargins', np.nan)
            })
        except Exception as e:
            warnings.warn(f'Fundamentals unavailable for {ticker}: {e}')

    if not fundamentals:
        raise ValueError('No fundamentals returned; retry the live snapshot separately.')
    df = pd.DataFrame(fundamentals).set_index('ticker')
    return df

print("Fetching fundamental data...")
fundamentals = get_fundamental_data(UNIVERSE)
print(f"Retrieved fundamentals for {len(fundamentals)} stocks")
print("\nSample fundamental data:")
print(fundamentals[['pe_ratio', 'pb_ratio', 'roe', 'market_cap']].head(10))

Fetching fundamental data...
Retrieved fundamentals for 52 stocks

Sample fundamental data:
         pe_ratio   pb_ratio      roe     market_cap
ticker                                              
AAPL    38.918484  46.057064  1.48751  4947135430656
MSFT    27.913746   8.421221  0.34039  3724727812096
GOOGL   17.819778   6.974419  0.48676  4341260091392
META    27.908321   7.230248  0.29848  1888326254592
NVDA    28.745892  23.977646  1.17211  5490544738304
AVGO    46.316730  17.367111  0.44245  1731204612096
CSCO    33.471474   8.746763  0.27319   439440736256
ORCL    23.285265   7.267391  0.41192   449206222848
JNJ     31.297330   7.637173  0.25742   649395372032
UNH     24.249197   3.470827  0.14153   338895896576


### Reading the computed output

Run the preceding cell for the current calculation. The old manuscript table has been removed because it was not generated by the corrected workflow and contained inconsistent rankings or arithmetic. The current fundamental snapshot is a mechanics demonstration; historical results use lagged price factors unless an as-reported fundamentals callback is supplied.


## AI Tutorial: Stock Selection and Portfolio Construction / Factor-Based Stock Ranking and Screening

### Code example 3
Source line: `Overleaf/Final Project/Chapter 7 - Equity Investing and Portfolio Management/main.tex`

In [4]:
def compute_momentum_factor(returns, lookback=12, skip=1):
    """
    Compute momentum factor: cumulative return over lookback months,
    skipping the most recent month to avoid reversal effects.

    Parameters:
    -----------
    returns : pd.DataFrame
        Monthly returns
    lookback : int
        Lookback period in months (default 12)
    skip : int
        Months to skip at end (default 1)

    Returns:
    --------
    pd.Series : Momentum scores for each stock
    """
    if lookback < 1 or skip < 0:
        raise ValueError('lookback must be positive and skip nonnegative.')
    if len(returns) < lookback + skip:
        return pd.Series(dtype=float)

    # Use returns from t-lookback-skip to t-skip
    momentum_returns = returns.iloc[-(lookback+skip):len(returns)-skip]

    # Cumulative return over the period
    cumulative = (1 + momentum_returns).prod(min_count=lookback) - 1

    return cumulative

def compute_volatility_factor(returns, lookback=12):
    """
    Compute volatility factor: annualized standard deviation.
    Lower volatility is considered better (negative factor).
    """
    if len(returns) < lookback:
        return pd.Series(dtype=float)

    recent_returns = returns.iloc[-lookback:]
    volatility = recent_returns.std() * np.sqrt(12)  # Annualize

    return -volatility  # Negative so higher score = lower vol

def rank_normalize(series):
    """
    Convert values to percentile ranks (0 to 1).
    Handles NaN values by excluding them from ranking.
    """
    return series.rank(pct=True, na_option='keep')

def compute_composite_score(fundamentals, monthly_returns, weights=None):
    """
    Compute composite factor score for stock ranking.

    Factors:
    - Value: Low P/E, low P/B (inverted and ranked)
    - Quality: High ROE, low debt/equity, high profit margin
    - Momentum: 12 monthly returns, excluding the most recent month (13-month history)
    - Low Volatility: Lower historical volatility

    Parameters:
    -----------
    fundamentals : pd.DataFrame
        Fundamental data
    monthly_returns : pd.DataFrame
        Monthly return history
    weights : dict, optional
        Factor weights (default: equal weight)

    Returns:
    --------
    pd.DataFrame : Factor scores and composite ranking
    """
    if weights is None:
        weights = {
            'value': 0.25,
            'quality': 0.25,
            'momentum': 0.30,
            'low_vol': 0.20
        }

    factor_names = ['value', 'quality', 'momentum', 'low_vol']
    if set(weights) != set(factor_names) or any(v < 0 for v in weights.values()) or not np.isclose(sum(weights.values()), 1):
        raise ValueError('Factor weights must be nonnegative and sum to one.')
    scores = pd.DataFrame(index=fundamentals.index)

    # Value factor: inverse of P/E and P/B (lower is better)
    # Nonpositive valuation ratios are not cheapness signals. Clipping reduces
    # outlier impact but ties all observations above each cap (information lost).
    pe_capped = fundamentals['pe_ratio'].where(fundamentals['pe_ratio'] > 0).clip(upper=100)
    pb_capped = fundamentals['pb_ratio'].where(fundamentals['pb_ratio'] > 0).clip(upper=50)

    scores['value_pe'] = rank_normalize(-pe_capped)
    scores['value_pb'] = rank_normalize(-pb_capped)
    scores['value'] = (scores['value_pe'] + scores['value_pb']) / 2

    # Quality factor: high ROE, low leverage, high margins
    scores['quality_roe'] = rank_normalize(fundamentals['roe'])
    scores['quality_debt'] = rank_normalize(-fundamentals['debt_to_equity'].where(
        fundamentals['debt_to_equity'] >= 0).clip(upper=500))
    scores['quality_margin'] = rank_normalize(fundamentals['profit_margin'])
    scores['quality'] = (scores['quality_roe'] + scores['quality_debt'] +
                         scores['quality_margin']) / 3

    # Momentum factor
    momentum = compute_momentum_factor(monthly_returns.drop(columns=[BENCHMARK],
                                       errors='ignore'))
    scores['momentum'] = rank_normalize(momentum)

    # Low volatility factor
    low_vol = compute_volatility_factor(monthly_returns.drop(columns=[BENCHMARK],
                                        errors='ignore'))
    scores['low_vol'] = rank_normalize(low_vol)

    # Composite score (weighted average of factor ranks)
    scores['composite'] = (
        weights['value'] * scores['value'].fillna(0.5) +
        weights['quality'] * scores['quality'].fillna(0.5) +
        weights['momentum'] * scores['momentum'].fillna(0.5) +
        weights['low_vol'] * scores['low_vol'].fillna(0.5)
    )

    # Final rank
    scores['rank'] = scores['composite'].rank(ascending=False)

    return scores.sort_values('composite', ascending=False)

# Compute factor scores
# This snapshot mixes live fundamentals with the fixed historical price window.
# It demonstrates ranking mechanics, not a historical trade recommendation.
ranking_universe = fundamentals.index.intersection(monthly_returns.columns)
ranking_universe = ranking_universe[monthly_returns[ranking_universe].tail(13).notna().all()]
factor_scores = compute_composite_score(fundamentals.loc[ranking_universe], monthly_returns[ranking_universe])
print("Factor-Based Stock Ranking (Top 15):")
print(factor_scores[['value', 'quality', 'momentum', 'low_vol',
                     'composite', 'rank']].head(15).round(3))

Factor-Based Stock Ranking (Top 15):
       value  quality  momentum  low_vol  composite  rank
MSFT   0.399    0.828     0.885    0.769      0.726   1.0
GOOGL  0.597    0.918     0.846    0.385      0.709   2.0
LIN    0.441    0.563     0.769    0.981      0.678   3.0
JPM    0.849      NaN     0.750    0.558      0.674   4.0
META   0.449    0.761     1.000    0.288      0.660   5.0
AMZN   0.559    0.677     0.904    0.308      0.642   6.0
SO     0.732    0.317     0.673    0.808      0.626   7.0
MCD      NaN      NaN     0.615    0.923      0.619   8.0
NVDA   0.277    0.979     0.981    0.019      0.612   9.0
DUK    0.871    0.318     0.385    0.942      0.601  10.0
COP    0.860    0.558     0.404    0.596      0.595  11.0
UNH    0.611    0.355     0.558    0.904      0.590  12.0
ORCL   0.478    0.540     0.865    0.365      0.587  13.0
HON    0.816    0.558     0.250    0.827      0.584  14.0
GE     0.180    0.598     0.962    0.481      0.579  15.0


### Reading the computed output

Run the preceding cell for the current calculation. The old manuscript table has been removed because it was not generated by the corrected workflow and contained inconsistent rankings or arithmetic. The current fundamental snapshot is a mechanics demonstration; historical results use lagged price factors unless an as-reported fundamentals callback is supplied.


## AI Tutorial: Stock Selection and Portfolio Construction / Factor-Based Stock Ranking and Screening / Stock Selection: Top Quintile

### Code example 4
Source line: `Overleaf/Final Project/Chapter 7 - Equity Investing and Portfolio Management/main.tex`

In [5]:
def select_stocks(factor_scores, n_stocks=None, percentile=0.20):
    """
    Select top-ranked stocks based on factor scores.

    Parameters:
    -----------
    factor_scores : pd.DataFrame
        Factor scores with 'composite' column
    n_stocks : int, optional
        Number of stocks to select (overrides percentile)
    percentile : float
        Top percentile to select (default 0.20 = top 20%)

    Returns:
    --------
    list : Selected stock tickers
    """
    if not 0 < percentile <= 1 or (n_stocks is not None and n_stocks < 1):
        raise ValueError('Select a positive stock count or percentile in (0, 1].')
    sorted_scores = factor_scores.dropna(subset=['composite']).sort_values('composite', ascending=False)
    if sorted_scores.empty:
        raise ValueError('No eligible stocks to select.')

    if n_stocks is None:
        n_stocks = max(10, int(len(sorted_scores) * percentile))

    selected = sorted_scores.head(n_stocks).index.tolist()
    return selected

# Select top quintile
selected_stocks = select_stocks(factor_scores, percentile=0.20)
print(f"\nSelected Portfolio ({len(selected_stocks)} stocks):")
print(selected_stocks)

# Get factor exposures for selected stocks
selected_scores = factor_scores.loc[selected_stocks]
print(f"\nAverage factor scores for selected portfolio:")
print(selected_scores[['value', 'quality', 'momentum', 'low_vol']].mean().round(3))

print(f"\nAverage factor scores for full universe:")
print(factor_scores[['value', 'quality', 'momentum', 'low_vol']].mean().round(3))


Selected Portfolio (10 stocks):
['MSFT', 'GOOGL', 'LIN', 'JPM', 'META', 'AMZN', 'SO', 'MCD', 'NVDA', 'DUK']

Average factor scores for selected portfolio:
value       0.575
quality     0.670
momentum    0.781
low_vol     0.598
dtype: float64

Average factor scores for full universe:
value       0.517
quality     0.512
momentum    0.510
low_vol     0.510
dtype: float64


### Reading the computed output

Run the preceding cell for the current calculation. The old manuscript table has been removed because it was not generated by the corrected workflow and contained inconsistent rankings or arithmetic. The current fundamental snapshot is a mechanics demonstration; historical results use lagged price factors unless an as-reported fundamentals callback is supplied.


## Continue in Chapter 9

Portfolio optimization, backtesting, performance evaluation and production considerations continue in the Chapter 9 notebook (`../chapter-09-ai-in-portfolio-optimization`).
